# 노트북 9. Tool Calling + LangChain/LangGraph 연계

> Phase 3 — 실전 기법: 챗봇을 똑똑하게 만드는 기술

LLM은 세상의 정보를 알지 못하고, 계산도 못 하고, API도 호출 못 합니다.
Tool Calling은 LLM에게 '손과 발'을 달아주는 메커니즘입니다.

**학습 목표**
- Tool Calling의 본질(의도 → 실행 → 결과 주입 → 답변 루프)을 이해한다
- google-genai에서 수동/자동 Function Calling을 구현할 수 있다
- LangChain의 @tool, bind_tools, ToolMessage 패턴을 활용할 수 있다
- LangGraph ToolNode로 자동화된 도구 실행 그래프를 구성할 수 있다
- 다중 도구 바인딩과 병렬 호출을 구현할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | Tool Calling 원리 + 3가지 구현 방식 | 읽고 실행 |
| Part 2 — 실습 | 수동/자동 Function Calling + ToolNode | 직접 작성 |
| Part 3 — 챌린지 | 다중 도구 도우미 봇 설계 | 강사와 함께 |

In [1]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langgraph

import os
import json
from google import genai
from google.genai import types

# API 키 설정 (Colab 환경 기준)
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

print("환경 설정 완료")

환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 Tool Calling의 본질

LLM은 다음을 **할 수 없습니다**:
- 실시간 날씨, 주가, 뉴스 검색
- 정확한 수학 계산 (큰 수, 소수점)
- 외부 API 호출 (이메일 발송, DB 조회)

Tool Calling은 이 한계를 극복하는 메커니즘입니다.

**핵심 루프: 4단계**

```
1. 사용자 질문 → 모델에 전달 (+ 도구 목록)
2. 모델이 "이 함수를 이 인자로 호출해야 한다"는 의도(intent)를 JSON으로 출력
3. 클라이언트(개발자 코드)가 실제 함수를 실행 → 결과를 모델에 다시 전달
4. 모델이 함수 결과를 바탕으로 최종 답변 생성
```

모델은 함수를 **직접 실행하지 않습니다**. "이 함수를 호출하세요"라는 **요청**만 합니다.

```
사용자: "서울 날씨 알려줘"
        ↓
모델: function_call(name="get_weather", args={"city": "서울"})
        ↓
클라이언트: get_weather("서울") → {"temp": 15, "condition": "맑음"}
        ↓
모델: "서울의 현재 날씨는 15도이며 맑습니다."
```

### Tool Calling이 없는 경우 vs 있는 경우

| 상황 | Tool Calling 없이 | Tool Calling 있으면 |
|------|------------------|-------------------|
| "서울 날씨 알려줘" | "서울은 사계절이 뚜렷한 도시로..." (학습 데이터 기반 추측) | get_weather("서울") → 실시간 데이터 → 정확한 답변 |
| "127 x 389는?" | "약 49,000 정도입니다" (근사값, 오류 가능) | calculator("127*389") → 49403 (정확한 결과) |
| "이메일 보내줘" | "죄송합니다, 이메일을 보낼 수 없습니다" | send_email(to=..., body=...) → 실제 발송 |

Tool Calling은 LLM의 **지식 한계**와 **실행 능력 부재**를 동시에 해결합니다.

## 1.2 google-genai: FunctionDeclaration

google-genai에서 도구를 정의하려면 `FunctionDeclaration`을 사용합니다.
도구의 이름, 설명, 파라미터를 JSON Schema 형식으로 정의합니다.

```python
from google.genai import types

get_weather_func = types.FunctionDeclaration(
    name="get_weather",
    description="지정된 도시의 현재 날씨를 조회합니다",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름"},
        },
        "required": ["city"],
    },
)
```

In [2]:
# 도구 정의: 날씨 조회 + 계산기
get_weather_func = types.FunctionDeclaration(
    name="get_weather",
    description="지정된 도시의 현재 날씨 정보를 조회합니다",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름 (예: 서울, 부산)"},
        },
        "required": ["city"],
    },
)

calculator_func = types.FunctionDeclaration(
    name="calculator",
    description="수학 계산을 수행합니다. 사칙연산, 거듭제곱 등을 지원합니다",
    parameters={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "계산할 수식 (예: 2+3, 15*7)"},
        },
        "required": ["expression"],
    },
)

# Tool 객체로 감싸기
tools = types.Tool(function_declarations=[get_weather_func, calculator_func])

print(f"도구 수: {len(tools.function_declarations)}")
for fd in tools.function_declarations:
    print(f"  - {fd.name}: {fd.description}")

도구 수: 2
  - get_weather: 지정된 도시의 현재 날씨 정보를 조회합니다
  - calculator: 수학 계산을 수행합니다. 사칙연산, 거듭제곱 등을 지원합니다


## 1.3 google-genai: 수동 Function Calling

수동 Function Calling의 전체 루프를 직접 구현합니다.
각 단계를 명시적으로 처리하면 동작 원리를 정확히 이해할 수 있습니다.

In [3]:
# 실제 함수 구현 (도구가 호출되었을 때 실행할 코드)
def get_weather(city: str) -> dict:
    """도시의 날씨를 반환합니다 (데모용 고정값)."""
    weather_data = {
        "서울": {"temp": 15, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 18, "condition": "흐림", "humidity": 65},
        "제주": {"temp": 20, "condition": "비", "humidity": 80},
    }
    return weather_data.get(city, {"temp": 0, "condition": "알 수 없음", "humidity": 0})

def calculator(expression: str) -> dict:
    """수식을 계산합니다."""
    try:
        result = eval(expression)  # 교육 목적 — 프로덕션에서는 안전한 파서 사용
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}

# 함수 매핑
available_functions = {
    "get_weather": get_weather,
    "calculator": calculator,
}

print("함수 준비 완료")
print(f"get_weather('서울') → {get_weather('서울')}")
print(f"calculator('15 * 7') → {calculator('15 * 7')}")

함수 준비 완료
get_weather('서울') → {'temp': 15, 'condition': '맑음', 'humidity': 45}
calculator('15 * 7') → {'result': 105}


In [4]:
# 1단계: 모델에 도구와 함께 질문 전달
response = client.models.generate_content(
    model=MODEL,
    contents="서울 날씨 알려줘",
    config=types.GenerateContentConfig(
        tools=[tools],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

# 2단계: 모델의 응답 확인 — function_call이 있는지 확인
print("=== 모델 응답 ===" )
for part in response.candidates[0].content.parts:
    if part.function_call:
        fc = part.function_call
        print(f"  function_call 발견!")
        print(f"  name: {fc.name}")
        print(f"  args: {dict(fc.args)}")
    elif part.text:
        print(f"  text: {part.text}")

=== 모델 응답 ===
  function_call 발견!
  name: get_weather
  args: {'city': '서울'}


In [5]:
# 3단계: 함수 실행 + 결과를 모델에 다시 전달
fc = response.candidates[0].content.parts[0].function_call
func_name = fc.name
func_args = dict(fc.args)

# 실제 함수 실행
func_result = available_functions[func_name](**func_args)
print(f"함수 실행: {func_name}({func_args}) → {func_result}")

# FunctionResponse 생성
function_response = types.Part.from_function_response(
    name=func_name,
    response=func_result,
)

# 4단계: 전체 대화 기록 + 함수 결과로 최종 답변 요청
final_response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Content(role="user", parts=[types.Part.from_text(text="서울 날씨 알려줘")]),
        response.candidates[0].content,  # 모델의 function_call 응답
        types.Content(role="user", parts=[function_response]),  # 함수 실행 결과
    ],
    config=types.GenerateContentConfig(
        tools=[tools],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

print(f"\n=== 최종 답변 ===")
print(final_response.text)

함수 실행: get_weather({'city': '서울'}) → {'temp': 15, 'condition': '맑음', 'humidity': 45}

=== 최종 답변 ===
서울은 현재 맑고, 기온은 15도이며 습도는 45%입니다.


> 수동 Function Calling은 4단계를 모두 직접 제어합니다.
> 함수 실행 전에 인자를 검증하거나, 결과를 가공하는 등 세밀한 제어가 가능합니다.
> 반면 코드가 길어지고, 루프를 직접 관리해야 하는 단점이 있습니다.

In [6]:
# 수동 Function Calling을 헬퍼 함수로 감싸기
# 실제 프로젝트에서는 이런 유틸리티를 만들어 재사용합니다

def manual_function_call(client, model, prompt, tools_obj, func_map, max_loops=5):
    """수동 function calling의 전체 루프를 실행합니다."""
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    config = types.GenerateContentConfig(
        tools=[tools_obj],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )

    for i in range(max_loops):
        response = client.models.generate_content(
            model=model, contents=contents, config=config,
        )

        # function_call이 없으면 최종 답변
        parts = response.candidates[0].content.parts
        has_fc = any(p.function_call for p in parts)
        if not has_fc:
            return response.text

        # function_call 실행
        contents.append(response.candidates[0].content)
        for part in parts:
            if part.function_call:
                fc = part.function_call
                result = func_map[fc.name](**dict(fc.args))
                fr = types.Part.from_function_response(name=fc.name, response=result)
                contents.append(types.Content(role="user", parts=[fr]))
                print(f"  루프 {i+1}: {fc.name}({dict(fc.args)})")

    return "최대 루프 초과"

# 테스트
answer = manual_function_call(
    client, MODEL, "제주 날씨 어때?", tools, available_functions,
)
print(f"\n최종 답변: {answer}")

  루프 1: get_weather({'city': '제주'})

최종 답변: 현재 제주는 비가 오고, 습도는 80%, 온도는 20도입니다.


## 1.4 google-genai: 자동 Function Calling

파이썬 함수를 `tools` 매개변수에 직접 전달하면,
SDK가 스키마 추출 → function_call 파싱 → 함수 실행 → 결과 주입을 **자동으로** 수행합니다.

함수의 **이름**, **docstring**, **타입 힌트**가 FunctionDeclaration으로 자동 변환됩니다.

In [7]:
# 자동 function calling — 파이썬 함수를 직접 전달
def get_weather_auto(city: str) -> dict:
    """지정된 도시의 현재 날씨 정보를 조회합니다.

    Args:
        city: 도시 이름 (예: 서울, 부산, 제주)

    Returns:
        온도, 날씨 상태, 습도를 포함하는 딕셔너리
    """
    weather_data = {
        "서울": {"temp": 15, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 18, "condition": "흐림", "humidity": 65},
        "제주": {"temp": 20, "condition": "비", "humidity": 80},
    }
    return weather_data.get(city, {"temp": 0, "condition": "알 수 없음", "humidity": 0})

# 함수를 직접 tools에 전달 — SDK가 나머지를 자동 처리
response = client.models.generate_content(
    model=MODEL,
    contents="부산 날씨 알려줘",
    config=types.GenerateContentConfig(
        tools=[get_weather_auto],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

# 자동으로 함수 실행 → 결과 주입 → 최종 답변까지 완료
print("=== 자동 Function Calling 결과 ===")
print(response.text)

=== 자동 Function Calling 결과 ===
부산은 현재 흐리고, 온도는 18도, 습도는 65%입니다.


In [8]:
# 자동 function calling에서 여러 함수를 동시에 전달
def calculator_auto(expression: str) -> dict:
    """수학 계산을 수행합니다.

    Args:
        expression: 계산할 수식 (예: 2+3, 100/7)

    Returns:
        계산 결과를 포함하는 딕셔너리
    """
    try:
        return {"result": eval(expression)}
    except Exception as e:
        return {"error": str(e)}

# 여러 함수를 리스트로 전달
response = client.models.generate_content(
    model=MODEL,
    contents="서울 날씨를 알려주고, 15 * 23도 계산해줘",
    config=types.GenerateContentConfig(
        tools=[get_weather_auto, calculator_auto],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

print("=== 여러 함수 자동 호출 결과 ===")
print(response.text)

=== 여러 함수 자동 호출 결과 ===
서울은 현재 맑고 온도는 15도이며 습도는 45%입니다. 15 * 23의 계산 결과는 345입니다.


> **자동 vs 수동 선택 기준**
> - **자동**: 빠른 프로토타이핑, 간단한 도구, 인자 검증이 불필요한 경우
> - **수동**: 인자 검증, 로깅, 에러 처리, 사용자 확인이 필요한 경우
> - 자동 function calling은 최대 **10회** 루프를 돕니다 (무한 루프 방지)

## 1.5 LangChain: @tool 데코레이터

LangChain에서는 `@tool` 데코레이터로 도구를 정의합니다.
함수의 **docstring**이 도구의 설명(description)이 됩니다.
타입 힌트가 파라미터 스키마로 변환됩니다.

In [9]:
from langchain_core.tools import tool

@tool
def get_weather_lc(city: str) -> str:
    """지정된 도시의 현재 날씨 정보를 조회합니다."""
    weather_data = {
        "서울": {"temp": 15, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 18, "condition": "흐림", "humidity": 65},
        "제주": {"temp": 20, "condition": "비", "humidity": 80},
    }
    data = weather_data.get(city, {"temp": 0, "condition": "알 수 없음", "humidity": 0})
    return json.dumps(data, ensure_ascii=False)

@tool
def calculator_lc(expression: str) -> str:
    """수학 계산을 수행합니다. 사칙연산, 거듭제곱 등을 지원합니다."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"

# 도구 속성 확인
print(f"name: {get_weather_lc.name}")
print(f"description: {get_weather_lc.description}")
print(f"args_schema: {get_weather_lc.args_schema.model_json_schema()}")
print()
print(f"name: {calculator_lc.name}")
print(f"description: {calculator_lc.description}")

name: get_weather_lc
description: 지정된 도시의 현재 날씨 정보를 조회합니다.
args_schema: {'description': '지정된 도시의 현재 날씨 정보를 조회합니다.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_lc', 'type': 'object'}

name: calculator_lc
description: 수학 계산을 수행합니다. 사칙연산, 거듭제곱 등을 지원합니다.


> **@tool 데코레이터 규칙**
> - **함수 이름**이 도구의 `name`이 됩니다 → 명확한 이름을 사용하세요
> - **docstring 첫 줄**이 도구의 `description`이 됩니다 → 가장 중요한 정보를 첫 줄에
> - **타입 힌트**가 파라미터 스키마로 변환됩니다 → 반드시 작성하세요
> - 반환 타입은 `str`을 권장합니다. JSON 문자열로 반환하면 모델이 파싱하기 쉽습니다

## 1.6 LangChain: bind_tools + ToolMessage 루프

`bind_tools()`로 모델에 도구를 바인딩하면,
모델은 필요할 때 도구 호출 의도를 `AIMessage.tool_calls`에 담아 반환합니다.

개발자는 이 의도를 파싱하여 함수를 실행하고,
결과를 `ToolMessage`로 모델에 다시 전달합니다.

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, ToolMessage

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
)

# 1단계: 모델에 도구 바인딩
llm_with_tools = llm.bind_tools([get_weather_lc, calculator_lc])

# 2단계: 모델 호출 — 도구 호출 의도가 포함된 AIMessage 반환
ai_message = llm_with_tools.invoke("제주 날씨 알려줘")

print(f"content: {ai_message.content!r}")
print(f"tool_calls: {ai_message.tool_calls}")

content: ''
tool_calls: [{'name': 'get_weather_lc', 'args': {'city': '제주'}, 'id': 'd1f1d0e3-3a77-4515-8d98-9e857d64ea68', 'type': 'tool_call'}]


In [11]:
# 3단계: 도구 실행 + ToolMessage 생성
tool_map = {
    "get_weather_lc": get_weather_lc,
    "calculator_lc": calculator_lc,
}

tool_messages = []
for tc in ai_message.tool_calls:
    func = tool_map[tc["name"]]
    result = func.invoke(tc["args"])
    tool_msg = ToolMessage(content=result, tool_call_id=tc["id"])
    tool_messages.append(tool_msg)
    print(f"실행: {tc['name']}({tc['args']}) → {result}")

# 4단계: 전체 대화 기록으로 최종 답변 요청
messages = [
    HumanMessage(content="제주 날씨 알려줘"),
    ai_message,       # 모델의 tool_call 응답
    *tool_messages,    # 도구 실행 결과
]

final = llm_with_tools.invoke(messages)
print(f"\n=== 최종 답변 ===")
print(final.content)

실행: get_weather_lc({'city': '제주'}) → {"temp": 20, "condition": "비", "humidity": 80}

=== 최종 답변 ===
제주에는 비가 오고 있으며, 습도는 80%, 온도는 20도입니다.


> `tool_calls`의 구조:
> ```python
> [
>     {
>         "name": "get_weather_lc",   # 도구 이름
>         "args": {"city": "제주"},     # 인자
>         "id": "call_abc123",          # 고유 ID (ToolMessage에서 매칭)
>     }
> ]
> ```
> `ToolMessage`의 `tool_call_id`는 반드시 해당 `tool_call`의 `id`와 일치해야 합니다.

## tool_call_id 매칭 규칙

`AIMessage`가 tool call을 요청하면, 그 결과를 담는 `ToolMessage`의 `tool_call_id`는 **반드시 원래 요청의 `id`와 동일**해야 합니다.

### 잘못된 예

```python
# AIMessage.tool_calls[0]['id'] = 'call_abc123' 인 상황에서
ToolMessage(tool_call_id="wrong_id")  # ❌ 모델이 결과를 매칭하지 못함
```

### 올바른 예

```python
for tc in ai_message.tool_calls:
    result = execute_tool(tc)
    ToolMessage(
        content=result,
        tool_call_id=tc["id"]  # ✅ 반드시 동일한 id 사용
    )
```

> **핵심**: `tool_call_id`가 일치하지 않으면 LLM이 어떤 tool call에 대한 응답인지 알 수 없어 오류가 발생하거나 무시됩니다.

## 1.7 LangGraph: ToolNode 소개

LangChain의 수동 루프를 자동화한 것이 LangGraph의 `ToolNode`입니다.

`ToolNode`는 다음을 자동으로 처리합니다:
1. `AIMessage.tool_calls`에서 도구 이름과 인자를 파싱
2. 해당 도구를 실행
3. 결과를 `ToolMessage`로 감싸서 반환

```python
from langgraph.prebuilt import ToolNode

tool_node = ToolNode([get_weather_lc, calculator_lc])
```

In [29]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

# ToolNode 생성
tool_node = ToolNode([get_weather_lc, calculator_lc])

# langgraph 1.0.x에서는 ToolNode를 직접 invoke할 수 없으므로
# 간단한 그래프로 감싸서 실행
builder = StateGraph(MessagesState)
builder.add_node("tools", tool_node)
builder.add_edge(START, "tools")
builder.add_edge("tools", END)
tool_graph = builder.compile()

# 테스트: 이전에 받은 ai_message를 전달
result = tool_graph.invoke({"messages": [ai_message]})

print("=== ToolNode 출력 ===")
for msg in result["messages"]:
    if hasattr(msg, "tool_call_id") and msg.tool_call_id:
        print(f"  type: {type(msg).__name__}")
        print(f"  content: {msg.content}")
        print(f"  tool_call_id: {msg.tool_call_id}")

=== ToolNode 출력 ===
  type: ToolMessage
  content: {"temp": 20, "condition": "비", "humidity": 80}
  tool_call_id: d1f1d0e3-3a77-4515-8d98-9e857d64ea68


## 1.8 LangGraph: 조건부 엣지와 도구 실행 그래프

ToolNode를 StateGraph에 통합하면 완전한 도구 호출 루프를 구성할 수 있습니다.

```
START → llm_node → [tool_calls 있음?] → Yes → tool_node → llm_node → ...
                                        → No  → END
```

LangGraph는 `tools_condition` 함수를 제공합니다.
이 함수는 마지막 메시지에 `tool_calls`가 있으면 `"tools"`를, 없으면 `"__end__"`를 반환합니다.

In [13]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import tools_condition

# LLM 노드: 도구가 바인딩된 모델 호출
def llm_node(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 그래프 구성
graph_builder = StateGraph(MessagesState)

# 노드 추가
graph_builder.add_node("llm", llm_node)
graph_builder.add_node("tools", ToolNode([get_weather_lc, calculator_lc]))

# 엣지 추가
graph_builder.add_edge(START, "llm")
graph_builder.add_conditional_edges("llm", tools_condition)  # tool_calls → tools, 없으면 → END
graph_builder.add_edge("tools", "llm")  # 도구 실행 후 다시 LLM으로

# 그래프 컴파일
graph = graph_builder.compile()

print("그래프 구성 완료")
print(f"노드: {list(graph.nodes.keys())}")

그래프 구성 완료
노드: ['__start__', 'llm', 'tools']


In [14]:
# 그래프 실행 — 전체 루프가 자동으로 동작
result = graph.invoke(
    {"messages": [HumanMessage(content="서울 날씨 알려줘")]}
)

print("=== 메시지 흐름 ===")
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  [{role}] tool_calls: {[tc['name'] for tc in msg.tool_calls]}")
    else:
        content = msg.content if isinstance(msg.content, str) else str(msg.content)[:100]
        print(f"  [{role}] {content[:80]}")

=== 메시지 흐름 ===
  [HumanMessage] 서울 날씨 알려줘
  [AIMessage] tool_calls: ['get_weather_lc']
  [ToolMessage] {"temp": 15, "condition": "맑음", "humidity": 45}
  [AIMessage] 서울은 현재 맑고, 기온은 15도이며, 습도는 45%입니다.


> LangGraph 그래프의 메시지 흐름:
> 1. `HumanMessage` → llm_node
> 2. `AIMessage(tool_calls=[...])` → tools_condition → tool_node
> 3. `ToolMessage(결과)` → llm_node
> 4. `AIMessage(최종 답변)` → tools_condition → END

### Checkpointer와 도구 호출 상태

LangGraph의  checkpointer를 사용하면,
도구 호출 이력을 포함한 전체 대화 상태를 유지할 수 있습니다.

In [15]:
from langgraph.checkpoint.memory import MemorySaver

# Checkpointer 추가
memory = MemorySaver()
graph_with_memory = graph_builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "tool-demo-1"}}

# 첫 번째 질문
r1 = graph_with_memory.invoke(
    {"messages": [HumanMessage(content="서울 날씨 알려줘")]},
    config=config,
)
print(f"1번째: {r1['messages'][-1].content[:80]}")

# 두 번째 질문 — 이전 대화 기록(도구 호출 포함)이 유지됨
r2 = graph_with_memory.invoke(
    {"messages": [HumanMessage(content="아까 말한 도시의 습도는 몇이었어?")]},
    config=config,
)
print(f"2번째: {r2['messages'][-1].content[:80]}")
print(f"\n총 메시지 수: {len(r2['messages'])} (도구 호출 이력 포함)")

1번째: 서울은 현재 맑고, 기온은 15도이며, 습도는 45%입니다.
2번째: 제가 이전에 알려드린 서울의 습도는 45%였습니다.

총 메시지 수: 6 (도구 호출 이력 포함)


> Checkpointer가 있으면 모델이 이전 도구 호출 결과를 기억합니다.
> "아까 검색한 결과에서..."와 같은 후속 질문에 정확히 답할 수 있습니다.

### 도구를 사용하지 않는 경우

도구가 바인딩되어 있어도, 모델이 도구 없이 답변할 수 있다고 판단하면
tool_calls 없이 직접 텍스트를 반환합니다.

In [16]:
# 도구가 필요 없는 질문
result_no_tool = graph.invoke(
    {"messages": [HumanMessage(content="안녕하세요, 반갑습니다!")]}
)

print("=== 도구 불필요 — 직접 답변 ===")
for msg in result_no_tool["messages"]:
    role = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  [{role}] tool_calls: {msg.tool_calls}")
    else:
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"  [{role}] {content[:80]}")

print(f"\nToolMessage 수: {sum(1 for m in result_no_tool['messages'] if type(m).__name__ == 'ToolMessage')}")
print("-> 모델이 도구 없이 직접 답변하여 tools_condition이 END로 분기")

=== 도구 불필요 — 직접 답변 ===
  [HumanMessage] 안녕하세요, 반갑습니다!
  [AIMessage] 안녕하세요! 무엇을 도와드릴까요?

ToolMessage 수: 0
-> 모델이 도구 없이 직접 답변하여 tools_condition이 END로 분기


> **도구 선택은 모델이 결정합니다**
> - 도구가 바인딩되어 있어도 항상 사용하는 것은 아닙니다
> - "안녕하세요"처럼 도구가 불필요한 질문에는 직접 답변합니다
> - "서울 날씨"처럼 도구가 필요한 질문에만 tool_calls를 생성합니다
> - 이 판단은 도구의 description과 사용자 질문의 의미적 유사도에 기반합니다

### Pydantic으로 도구 파라미터 정의

@tool 데코레이터에서 Pydantic BaseModel을 로 지정하면,
파라미터에 상세한 설명과 제약 조건을 추가할 수 있습니다.

In [17]:
from pydantic import BaseModel, Field

class HotelSearchInput(BaseModel):
    city: str = Field(description="검색할 도시 이름")
    budget: int = Field(description="1박 예산 (원 단위, 예: 100000)")
    stars: int = Field(default=3, description="최소 별점 (1~5)")

@tool(args_schema=HotelSearchInput)
def search_hotel(city: str, budget: int, stars: int = 3) -> str:
    """도시에서 예산에 맞는 호텔을 검색합니다."""
    return json.dumps({
        "hotel": f"{city} {stars}성급 호텔",
        "price": budget,
        "available": True,
    }, ensure_ascii=False)

print(f"name: {search_hotel.name}")
print(f"schema: {json.dumps(search_hotel.args_schema.model_json_schema(), indent=2, ensure_ascii=False)}")

name: search_hotel
schema: {
  "properties": {
    "city": {
      "description": "검색할 도시 이름",
      "title": "City",
      "type": "string"
    },
    "budget": {
      "description": "1박 예산 (원 단위, 예: 100000)",
      "title": "Budget",
      "type": "integer"
    },
    "stars": {
      "default": 3,
      "description": "최소 별점 (1~5)",
      "title": "Stars",
      "type": "integer"
    }
  },
  "required": [
    "city",
    "budget"
  ],
  "title": "HotelSearchInput",
  "type": "object"
}


> **args_schema를 사용하면**
> - 각 파라미터에 `Field(description=...)`을 추가하여 모델에게 더 정확한 인자 가이드를 제공합니다
> - 기본값, 타입 제약 등 Pydantic 검증이 자동으로 적용됩니다
> - 단순한 도구는 타입 힌트만으로 충분하지만, 복잡한 도구에는 args_schema가 효과적입니다

## 1.9 다중 도구와 병렬 호출

모델은 한 번의 응답에서 **여러 도구를 동시에** 호출할 수 있습니다.
예를 들어 "서울과 부산 날씨를 비교해줘"라는 질문에 대해
`get_weather("서울")`과 `get_weather("부산")`을 동시에 요청할 수 있습니다.

어떤 도구를 언제 쓸지는 **모델이 판단**합니다.
따라서 도구의 **description** 품질이 매우 중요합니다.

In [18]:
# 병렬 도구 호출 테스트
result = graph.invoke(
    {"messages": [HumanMessage(content="서울과 부산 날씨를 비교해줘")]}
)

print("=== 메시지 흐름 ===")
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  [{role}] tool_calls ({len(msg.tool_calls)}개):")
        for tc in msg.tool_calls:
            print(f"    - {tc['name']}({tc['args']})")
    elif isinstance(msg.content, str) and msg.content:
        print(f"  [{role}] {msg.content[:100]}")
    else:
        print(f"  [{role}] {str(msg.content)[:100]}")

=== 메시지 흐름 ===
  [HumanMessage] 서울과 부산 날씨를 비교해줘
  [AIMessage] tool_calls (2개):
    - get_weather_lc({'city': '서울'})
    - get_weather_lc({'city': '부산'})
  [ToolMessage] {"temp": 15, "condition": "맑음", "humidity": 45}
  [ToolMessage] {"temp": 18, "condition": "흐림", "humidity": 65}
  [AIMessage] 서울은 맑고 기온은 15도이며 습도는 45%입니다.

부산은 흐리고 기온은 18도이며 습도는 65%입니다.


In [19]:
# 서로 다른 도구를 동시에 호출하는 경우
result = graph.invoke(
    {"messages": [HumanMessage(content="서울 날씨를 알려주고, 15 * 23 + 47을 계산해줘")]}
)

print("=== 서로 다른 도구 동시 호출 ===")
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [{role}] {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"  [{role}] {msg.content}")
    elif isinstance(msg.content, str) and msg.content:
        print(f"  [{role}] {msg.content[:100]}")

=== 서로 다른 도구 동시 호출 ===
  [HumanMessage] 서울 날씨를 알려주고, 15 * 23 + 47을 계산해줘
  [AIMessage] get_weather_lc({'city': '서울'})
  [AIMessage] calculator_lc({'expression': '15 * 23 + 47'})
  [ToolMessage] {"temp": 15, "condition": "맑음", "humidity": 45}
  [ToolMessage] 392
  [AIMessage] 서울은 현재 맑고, 기온은 15도, 습도는 45%입니다.
15 * 23 + 47의 계산 결과는 392입니다.


## 1.10 도구 설명(description)의 중요성

모델이 어떤 도구를 선택할지는 **description**에 달려 있습니다.
설명이 부정확하면 모델은 잘못된 도구를 선택하거나, 도구를 전혀 사용하지 않습니다.

In [20]:
# 나쁜 description vs 좋은 description 비교

@tool
def bad_tool(x: str) -> str:
    """도구입니다."""
    return f"결과: {x}"

@tool
def good_tool(city: str) -> str:
    """지정된 도시의 현재 날씨 정보(온도, 날씨 상태, 습도)를 조회합니다. 한국 도시만 지원합니다."""
    return json.dumps({"temp": 15, "condition": "맑음"}, ensure_ascii=False)

print("=== 나쁜 description ===")
print(f"  name: {bad_tool.name}")
print(f"  description: {bad_tool.description}")
print(f"  → 모델이 이 도구가 무엇인지 판단할 수 없음")

print("\n=== 좋은 description ===")
print(f"  name: {good_tool.name}")
print(f"  description: {good_tool.description}")
print(f"  → 어떤 상황에서 사용하는지 명확")

=== 나쁜 description ===
  name: bad_tool
  description: 도구입니다.
  → 모델이 이 도구가 무엇인지 판단할 수 없음

=== 좋은 description ===
  name: good_tool
  description: 지정된 도시의 현재 날씨 정보(온도, 날씨 상태, 습도)를 조회합니다. 한국 도시만 지원합니다.
  → 어떤 상황에서 사용하는지 명확


> **좋은 도구 description 작성 원칙**
> - **무엇을 하는지** 명확히: "날씨를 조회합니다" (O), "도구입니다" (X)
> - **입력 조건**을 명시: "한국 도시만 지원" → 모델이 다른 나라 도시를 넣지 않음
> - **반환값**을 설명: "온도, 상태, 습도를 포함하는 JSON" → 모델이 후처리 계획을 세움
> - **제한사항**을 명시: "최대 10개 결과" → 모델이 기대치를 조절

## 1.11 google-genai vs LangChain vs LangGraph 비교

| 항목 | google-genai (수동) | google-genai (자동) | LangChain | LangGraph |
|------|-------------------|-------------------|-----------|----------|
| 도구 정의 | FunctionDeclaration | 파이썬 함수 | @tool | @tool |
| 루프 관리 | 직접 구현 | SDK 자동 | 직접 구현 | 그래프 자동 |
| 상태 관리 | contents 리스트 | SDK 내부 | messages 리스트 | MessagesState |
| 병렬 호출 | 직접 파싱 | 자동 | 직접 파싱 | ToolNode 자동 |
| 확장성 | 낮음 | 중간 | 중간 | 높음 |
| 적합한 경우 | 원리 학습 | 빠른 프로토타입 | 커스텀 로직 | 프로덕션 에이전트 |

---

### 생각해보기

1. 모델이 도구를 직접 실행하지 않고 "의도"만 표현하는 설계는 어떤 이점이 있을까요? 보안, 비용, 유연성 측면에서 생각해보세요.
2. 자동 function calling은 편리하지만, 어떤 상황에서 수동 function calling이 더 적합할까요? 예를 들어 결제 API를 호출하는 경우를 생각해보세요.
3. 도구가 10개 이상이면 모델의 도구 선택 정확도가 떨어질 수 있습니다. 이를 어떻게 해결할 수 있을까요?

> **병렬 호출의 이점**
> - 네트워크 요청이 필요한 도구(API 호출 등)를 동시에 실행하면 응답 시간이 단축됩니다
> - ToolNode는 병렬로 도착한 tool_calls를 모두 실행한 뒤 결과를 한 번에 반환합니다
> - 단, 도구 간 의존성(A의 결과가 B의 입력)이 있으면 순차 호출이 필요합니다

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 9-1: google-genai 수동 Function Calling

google-genai로 수동 function calling의 전체 루프를 구현하세요.

**요구사항**
1. `search_book` 도구를 FunctionDeclaration으로 정의 (파라미터: title(str))
2. 실제 함수 구현 (데모용 고정 데이터)
3. "해리포터 책 검색해줘"로 호출
4. function_call 파싱 → 함수 실행 → FunctionResponse → 최종 답변

In [21]:
# TODO: google-genai 수동 function calling

# 정답
def search_book(title: str) -> dict:
    books = {
        "해리포터": {"title": "해리포터와 마법사의 돌", "author": "J.K. 롤링", "year": 1997},
        "반지의 제왕": {"title": "반지의 제왕", "author": "J.R.R. 톨킨", "year": 1954},
    }
    for key, val in books.items():
        if key in title:
            return val
    return {"error": "책을 찾을 수 없습니다"}

search_book_func = types.FunctionDeclaration(
    name="search_book",
    description="책 제목으로 도서 정보를 검색합니다",
    parameters={
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "검색할 책 제목"},
        },
        "required": ["title"],
    },
)
book_tools = types.Tool(function_declarations=[search_book_func])

resp = client.models.generate_content(
    model=MODEL,
    contents="해리포터 책 검색해줘",
    config=types.GenerateContentConfig(
        tools=[book_tools],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

fc = resp.candidates[0].content.parts[0].function_call
func_result = search_book(**dict(fc.args))
print(f"함수 실행: {fc.name}({dict(fc.args)}) → {func_result}")

function_response = types.Part.from_function_response(
    name=fc.name, response=func_result,
)

final = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Content(role="user", parts=[types.Part.from_text(text="해리포터 책 검색해줘")]),
        resp.candidates[0].content,
        types.Content(role="user", parts=[function_response]),
    ],
    config=types.GenerateContentConfig(
        tools=[book_tools],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)
print(f"최종 답변: {final.text}")

print("TODO를 완성해주세요")

함수 실행: search_book({'title': '해리포터'}) → {'title': '해리포터와 마법사의 돌', 'author': 'J.K. 롤링', 'year': 1997}
최종 답변: J.K. 롤링의 1997년에 출판된 해리포터와 마법사의 돌을 찾았습니다.
TODO를 완성해주세요


### 실습 9-2: google-genai 자동 Function Calling

파이썬 함수를 직접 전달하여 자동 function calling을 구현하세요.

**요구사항**
1. `translate_text(text: str, target_lang: str) -> str` 함수 정의 (데모용 고정 번역)
2. docstring과 타입 힌트를 상세하게 작성
3. 함수를 tools에 직접 전달하여 자동 호출
4. "'안녕하세요'를 영어로 번역해줘"로 테스트

In [22]:
# TODO: google-genai 자동 function calling

# 정답
def translate_text(text: str, target_lang: str) -> dict:
    """텍스트를 지정된 언어로 번역합니다.

    Args:
        text: 번역할 원본 텍스트
        target_lang: 번역 대상 언어 (예: 영어, 일본어, 중국어)

    Returns:
        원본 텍스트와 번역 결과를 포함하는 딕셔너리
    """
    translations = {
        ("안녕하세요", "영어"): "Hello",
        ("안녕하세요", "일본어"): "こんにちは",
        ("감사합니다", "영어"): "Thank you",
    }
    result = translations.get((text, target_lang), f"[번역: {text} → {target_lang}]")
    return {"original": text, "translated": result, "target_lang": target_lang}

response = client.models.generate_content(
    model=MODEL,
    contents="'안녕하세요'를 영어로 번역해줘",
    config=types.GenerateContentConfig(
        tools=[translate_text],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)
print(response.text)

print("TODO를 완성해주세요")

"안녕하세요"를 영어로 번역하면 "Hello"입니다.
TODO를 완성해주세요


### 실습 9-3: LangChain @tool + bind_tools

LangChain의 @tool 데코레이터로 도구를 정의하고 모델에 바인딩하세요.

**요구사항**
1. `get_stock_price(symbol: str) -> str` 도구 정의 (데모용 고정 주가)
2. `llm.bind_tools()`로 모델에 바인딩
3. "삼성전자 주가 알려줘"로 호출
4. `ai_message.tool_calls`를 확인하여 도구 호출 의도 출력

In [23]:
# TODO: LangChain @tool + bind_tools

# 정답
@tool
def get_stock_price(symbol: str) -> str:
    """주식 종목의 현재 가격을 조회합니다. 한국 주식 종목명을 입력하세요."""
    stocks = {
        "삼성전자": {"price": 72500, "change": "+1.2%"},
        "SK하이닉스": {"price": 185000, "change": "-0.8%"},
        "카카오": {"price": 42300, "change": "+0.5%"},
    }
    data = stocks.get(symbol, {"error": f"{symbol} 종목을 찾을 수 없습니다"})
    return json.dumps(data, ensure_ascii=False)

llm_stock = llm.bind_tools([get_stock_price])
ai_msg = llm_stock.invoke("삼성전자 주가 알려줘")

print(f"content: {ai_msg.content!r}")
print(f"tool_calls: {ai_msg.tool_calls}")
for tc in ai_msg.tool_calls:
    print(f"  name: {tc['name']}, args: {tc['args']}, id: {tc['id']}")

print("TODO를 완성해주세요")

content: ''
tool_calls: [{'name': 'get_stock_price', 'args': {'symbol': '삼성전자'}, 'id': 'b7cd51df-2be4-4cdd-afba-cfb5c7675d1f', 'type': 'tool_call'}]
  name: get_stock_price, args: {'symbol': '삼성전자'}, id: b7cd51df-2be4-4cdd-afba-cfb5c7675d1f
TODO를 완성해주세요


### 실습 9-4: LangChain ToolMessage 루프

실습 9-3에서 받은 tool_calls를 실행하고 ToolMessage로 최종 답변을 받으세요.

**요구사항**
1. `ai_message.tool_calls`를 순회하며 각 도구를 실행
2. 실행 결과를 `ToolMessage`로 감싸기 (tool_call_id 매칭 필수)
3. 전체 메시지 리스트([HumanMessage, AIMessage, ToolMessage])로 최종 답변 요청
4. 최종 답변 출력

In [24]:
# TODO: ToolMessage 루프로 최종 답변 받기

# 정답
# # 실습 9-3의 결과를 사용합니다. 실습 9-3을 먼저 완성하세요.
tool_map = {"get_stock_price": get_stock_price}

tool_msgs = []
for tc in ai_msg.tool_calls:
    result = tool_map[tc["name"]].invoke(tc["args"])
    tool_msgs.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    print(f"실행: {tc['name']}({tc['args']}) → {result}")

messages = [
    HumanMessage(content="삼성전자 주가 알려줘"),
    ai_msg,
    *tool_msgs,
]

final = llm_stock.invoke(messages)
print(f"\n최종 답변: {final.content}")

print("TODO를 완성해주세요")

실행: get_stock_price({'symbol': '삼성전자'}) → {"price": 72500, "change": "+1.2%"}

최종 답변: 삼성전자의 현재 주가는 72500원이며, +1.2% 변동이 있었습니다.
TODO를 완성해주세요


### 실습 9-5: LangGraph ToolNode 그래프

LangGraph의 StateGraph + ToolNode로 자동화된 도구 실행 그래프를 구성하세요.

**요구사항**
1. `get_weather_lc`와 `calculator_lc` 도구 사용 (이론에서 정의한 것)
2. llm_node와 tool_node로 그래프 구성
3. `tools_condition`으로 조건부 엣지 설정
4. "15의 7승은?"으로 테스트 후 메시지 흐름 출력

In [25]:
# TODO: LangGraph ToolNode 그래프

# 정답
def my_llm_node(state: MessagesState):
    llm_tools = llm.bind_tools([get_weather_lc, calculator_lc])
    response = llm_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("llm", my_llm_node)
builder.add_node("tools", ToolNode([get_weather_lc, calculator_lc]))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")
my_graph = builder.compile()

result = my_graph.invoke(
    {"messages": [HumanMessage(content="15의 7승은?")]}
)

for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"[{role}] tool_calls: {[tc['name'] for tc in msg.tool_calls]}")
    else:
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"[{role}] {content[:100]}")

print("TODO를 완성해주세요")

[HumanMessage] 15의 7승은?
[AIMessage] tool_calls: ['calculator_lc']
[ToolMessage] 170859375
[AIMessage] 15의 7승은 170,859,375입니다.
TODO를 완성해주세요


### 실습 9-6: 다중 도구 병렬 호출

여러 도구가 바인딩된 그래프에서 병렬 호출을 테스트하세요.

**요구사항**
1. 이론에서 만든 그래프(`graph`) 사용
2. "서울과 제주 날씨를 비교하고, 두 도시 기온 차이를 계산해줘"로 호출
3. 각 단계의 메시지 타입과 내용을 출력
4. 총 도구 호출 횟수를 세기

In [26]:
# TODO: 다중 도구 병렬 호출

# 정답
result = graph.invoke(
    {"messages": [HumanMessage(content="서울과 제주 날씨를 비교하고, 두 도시 기온 차이를 계산해줘")]}
)

tool_call_count = 0
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] {tc['name']}({tc['args']})")
            tool_call_count += 1
    elif role == "ToolMessage":
        print(f"[{role}] {msg.content}")
    elif isinstance(msg.content, str) and msg.content:
        print(f"[{role}] {msg.content[:100]}")

print(f"\n총 도구 호출 횟수: {tool_call_count}")

print("TODO를 완성해주세요")

[HumanMessage] 서울과 제주 날씨를 비교하고, 두 도시 기온 차이를 계산해줘
[AIMessage] get_weather_lc({'city': '서울'})
[AIMessage] get_weather_lc({'city': '제주'})
[ToolMessage] {"temp": 15, "condition": "맑음", "humidity": 45}
[ToolMessage] {"temp": 20, "condition": "비", "humidity": 80}
[AIMessage] calculator_lc({'expression': '20 - 15'})
[ToolMessage] 5

총 도구 호출 횟수: 3
TODO를 완성해주세요


### 실습 9-7: 도구 설명 품질 실험

도구의 description을 변경하여 모델의 도구 선택에 미치는 영향을 확인하세요.

**요구사항**
1. 동일한 기능의 도구를 2가지 description으로 정의 (모호한 것 vs 명확한 것)
2. 두 도구를 각각 모델에 바인딩
3. 같은 프롬프트로 호출하여 tool_calls 비교
4. 모호한 description에서 도구가 호출되지 않는 경우를 확인

In [27]:
# TODO: 도구 설명 품질 실험

# 정답
@tool
def vague_tool(q: str) -> str:
    """무언가를 처리합니다."""
    return json.dumps({"temp": 15, "condition": "맑음"}, ensure_ascii=False)

@tool
def clear_tool(city: str) -> str:
    """지정된 도시의 실시간 날씨(온도, 상태, 습도)를 조회합니다. 한국 도시명을 입력하세요."""
    return json.dumps({"temp": 15, "condition": "맑음"}, ensure_ascii=False)

prompt = "서울 날씨가 어때?"

# 모호한 도구
vague_llm = llm.bind_tools([vague_tool])
vague_result = vague_llm.invoke(prompt)
print(f"모호한 도구 — tool_calls: {vague_result.tool_calls}")
print(f"  content: {vague_result.content[:100] if vague_result.content else '(없음)'}")

# 명확한 도구
clear_llm = llm.bind_tools([clear_tool])
clear_result = clear_llm.invoke(prompt)
print(f"\n명확한 도구 — tool_calls: {clear_result.tool_calls}")
print(f"  content: {clear_result.content[:100] if clear_result.content else '(없음)'}")

print("TODO를 완성해주세요")

모호한 도구 — tool_calls: []
  content: [{'type': 'text', 'text': '저는 서울 날씨에 대한 정보를 제공할 수 없습니다.', 'extras': {'signature': 'CuIEAb4+9vs9/FwAqR33ZdYrFfbRO5B+m6PeY6hSZRjYd4QT86Jv1J1v4dXAw/+4KFdaysJQn+ra9PfwcBQR2tbYO6h5zvWwkYmqwlJY8g9QvMQS3xNH5y7pSHHfBefsTJopOywg3LxDitlhb2k3JW0r/zEdjOdfX8wDcX+l78vZ2c3N+x8+CDOrGkbMcMpteehcq0vdFrTt1pnnbgAALCJkm3pKOo3FJrnjyIgOUGuC66heqW3B5ACOWjiMS97JcMWSALQV4SvaWeNna2+FgQXlhH2seUbJcEM4CUIMf92FPmenijzFFkNe7KtrlQv5ncWMon3CLfe7RtyTtmgTCRwkn9XMzRP1urTMrKkvRZ5rYu8AoR++49vkVOqFA/kwRHyEd6275bS//HK8n0wI16bG7ROlqvKKRDJmeTshucBmc55jptZ1iqSc+oEO440eSGC2rWpyYnUgaiySyKZWDrj269p1uRBtTpVgWRKe4B9p9xNKVXIoTfKxOzDVRfPwPYvi6a/SwlmXp4Hvp3GRdA0bU1IuItbeFlq8Lez9Mm0IzB1l9Ylikbe1ulzCQ4zhEi9NBG9WSHAk57yLvBH2j7MrkSlrV896EV9PkFBWJ1yfOsX178jMtXaDmtRB99JSysokZkJ6aQK4No8big79cR82W58T95ET2CbyZCULiORgFXomYkcDTHs9zyjsuUm/q+iZMM3W8ZqKZ/p2znafajTpyysq0FS9YEtfK+vnyqttjARSFtcmOD2NtSvpLt2PnyDElYu++OIbKhVAdz3BYdqTIIFjHzFp5AnJFMC3JXKbNTPqlQ=='}}]

명확한 도구 — tool_calls: [{'name': 'clear_tool', 'args': {'c

---

### 생각해보기

1. 실습 9-1에서 수동 루프와 실습 9-5의 LangGraph 그래프를 비교했을 때, 코드 양과 유지보수성에서 어떤 차이가 있었나요? 프로젝트 규모에 따라 어떤 방식을 선택하겠습니까?
2. 실습 9-6에서 모델이 병렬로 도구를 호출했다면, 도구 실행 순서가 결과에 영향을 줄 수 있을까요? 도구 간 의존성이 있는 경우 어떻게 처리해야 할까요?
3. 실습 9-7에서 모호한 description이 도구 선택에 미치는 영향을 확인했습니다. 도구가 20개 이상일 때 description 외에 도구 선택 정확도를 높이는 방법은 무엇일까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 9-1: 다중 도구를 활용하는 도우미 봇 설계 (난이도: 중)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

3개 이상의 도구를 가진 **여행 도우미 봇**을 LangGraph로 설계하세요.

**도구 요구사항**
- `get_weather(city)`: 도시의 날씨 조회
- `search_hotel(city, budget)`: 도시의 호텔 검색 (budget: 가격대)
- `get_exchange_rate(currency)`: 환율 조회 (예: USD, JPY)

**과제**
- 3개 도구를 @tool로 정의 (데모용 고정 데이터)
- LangGraph StateGraph + ToolNode로 그래프 구성
- "도쿄 여행을 계획하고 있어. 날씨, 호텔, 엔화 환율을 알려줘"로 테스트
- 메시지 흐름을 출력하여 어떤 도구가 어떤 순서로 호출되었는지 확인

**힌트**
- description에 어떤 상황에서 사용하는 도구인지 명확히 작성하세요
- 모델이 3개 도구를 모두 호출하도록 프롬프트를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

---

### 생각해보기

1. 여행 도우미 봇에서 도구 호출 순서가 중요할까요? 예를 들어 환율을 먼저 조회한 후 그 결과를 호텔 가격 변환에 사용해야 한다면, 현재 설계로 가능할까요?
2. 도구 실행이 실패하면(예: API 타임아웃) 현재 그래프는 어떻게 동작할까요? 에러 핸들링을 그래프에 추가하려면 어떻게 해야 할까요?
3. 사용자가 "여행 도우미 봇"에게 날씨와 무관한 질문(예: "2+3은?")을 하면 어떻게 해야 할까요? 도구 사용을 거부하는 설계가 필요할까요?